# 07b · Fine-Tuning the Language Models

**Goal.** A follow-up to notebook 07: instead of freezing each backbone, **fine-tune** (further train) it
on our task and see if that helps.

**Why it matters.** Fine-tuning can boost accuracy when there is enough data — this notebook measures
whether that is the case here, and which backbone benefits most.

**Output:** `data/stage2_semantic/backbone_benchmark_finetuned.json`.

## 0. Environment & dependencies (+ load HF_TOKEN from .env for the gated IndicBERT)

In [ ]:
import os, re, sys, json, gc, time, logging, warnings
from pathlib import Path

for _stream in ("stdout", "stderr"):
    try:
        getattr(sys, _stream).reconfigure(encoding="utf-8")
    except Exception:
        pass

import numpy as np

def _try(name):
    try:
        __import__(name); return True
    except Exception:
        return False

assert _try("torch") and _try("transformers") and _try("sklearn"), "needs torch+transformers+sklearn"

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("ft")
PART_VERSION = "1.0"
SEED = 42

# load HF_TOKEN from model_implementation/.env WITHOUT printing its value (gated IndicBERT)
def load_env_token(root):
    envf = root / ".env"
    if envf.exists():
        for line in envf.read_text(encoding="utf-8", errors="ignore").splitlines():
            line = line.strip()
            if line and not line.startswith("#") and line.split("=", 1)[0].strip() == "HF_TOKEN":
                val = line.split("=", 1)[1].strip().strip('"').strip("'")
                if val:
                    os.environ["HF_TOKEN"] = val
                    return True
    return False

## 1. Configuration

In [ ]:
import torch, random
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

def find_project_root():
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / "data" / "stage2_semantic").exists() or (d / ".env").exists():
            return d
    return here

PROJECT_ROOT = find_project_root()
SEM_DIR = PROJECT_ROOT / "data" / "stage2_semantic"
OUTPUT_DIR = SEM_DIR
HAS_TOKEN = load_env_token(PROJECT_ROOT)

ALL_BACKBONES = [
    ("muril",      "google/muril-base-cased",         False),
    ("indicbert",  "ai4bharat/indic-bert",            True),   # gated -> needs access GRANT (token alone insufficient)
    ("indicbertv2","ai4bharat/IndicBERTv2-MLM-only",  False),  # UNGATED substitute for the gated indic-bert
    ("xlmr",       "xlm-roberta-base",                 False),
]
RUN_ONLY = [b.strip() for b in os.environ.get("RUN_ONLY", "").split(",") if b.strip()]

TOKEN_CAPS = [int(c) for c in os.environ.get("FT_TOKEN_CAPS", "128,256").split(",") if c.strip()]
MAX_FT_ROWS = int(os.environ.get("MAX_FT_ROWS", "0"))   # >0 -> stratified subsample TRAIN (CPU scale-up cap, D-SU3)
EPOCHS = 4
LR = 2e-5
BATCH = 8
EVAL_BATCH = 32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TIER_NAMES = {0: "Neutral", 1: "Stress", 2: "Moderate", 3: "Crisis"}

log.info("device=%s | HF_TOKEN loaded=%s | epochs=%d lr=%.0e batch=%d | RUN_ONLY=%s",
         DEVICE, HAS_TOKEN, EPOCHS, LR, BATCH, RUN_ONLY or "ALL")

## 2. Load corpus (TRAIN=youtube+synthetic, TEST=reddit+twitter frozen)

In [ ]:
def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

train_rows = load_jsonl(SEM_DIR / "corpus_preprocessed_train.jsonl")
test_rows  = load_jsonl(SEM_DIR / "corpus_preprocessed_test.jsonl")

if MAX_FT_ROWS and len(train_rows) > MAX_FT_ROWS:   # D-SU3: cap CPU fine-tune cost at scale (stratified, seeded)
    import random as _rnd; _rnd.seed(42)
    by_t = {}
    for r in train_rows: by_t.setdefault(int(r["tier"]), []).append(r)
    per = max(1, MAX_FT_ROWS // 4); sub = []
    for t, rs in by_t.items():
        _rnd.shuffle(rs); sub += rs[:per]
    _rnd.shuffle(sub); train_rows = sub
    log.info("MAX_FT_ROWS=%d -> subsampled TRAIN to %d (stratified, seed 42)", MAX_FT_ROWS, len(train_rows))

def texts_of(rows):
    return [str(r.get("text_muril") or r.get("post_text_clean") or "") for r in rows]

X_train_txt, y_train = texts_of(train_rows), np.array([int(r["tier"]) for r in train_rows])
X_test_txt,  y_test  = texts_of(test_rows),  np.array([int(r["tier"]) for r in test_rows])
assert set(r["platform"] for r in train_rows) <= {"youtube", "synthetic"}, "TRAIN platform leak!"
assert set(r["platform"] for r in test_rows)  <= {"reddit", "twitter"},   "TEST platform leak!"
log.info("train=%d %s | test=%d %s", len(train_rows),
         {int(t): int((y_train == t).sum()) for t in range(4)}, len(test_rows),
         {int(t): int((y_test == t).sum()) for t in range(4)})

## 3. Fine-tune + evaluate one (backbone, token-cap)

AutoModelForSequenceClassification (4 tiers), class-weighted CrossEntropy (inverse-frequency from TRAIN),
AdamW, `EPOCHS` epochs, batch `BATCH`. Evaluate on the frozen TEST: accuracy, macro-F1, MCC, per-tier F1,
Tier-3 recall. Deterministic seeding for reproducibility.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score, accuracy_score, recall_score, matthews_corrcoef

def finetune_eval(model_id, cap, gated):
    tkw = {"token": os.environ["HF_TOKEN"]} if (gated and os.environ.get("HF_TOKEN")) else {}
    tok = AutoTokenizer.from_pretrained(model_id, **tkw) if _try_tok(model_id, tkw) else None
    if tok is None:
        tok = AutoTokenizer.from_pretrained(model_id, use_fast=False, **tkw)
    model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=4, **tkw).to(DEVICE)

    enc_tr = tok(X_train_txt, padding=True, truncation=True, max_length=cap, return_tensors="pt")
    enc_te = tok(X_test_txt,  padding=True, truncation=True, max_length=cap, return_tensors="pt")
    ds = TensorDataset(enc_tr["input_ids"], enc_tr["attention_mask"], torch.tensor(y_train))
    g = torch.Generator(); g.manual_seed(SEED)
    dl = DataLoader(ds, batch_size=BATCH, shuffle=True, generator=g)

    counts = np.bincount(y_train, minlength=4).astype(float)
    w = torch.tensor(counts.sum() / (4.0 * np.clip(counts, 1, None)), dtype=torch.float).to(DEVICE)
    lossfn = torch.nn.CrossEntropyLoss(weight=w)
    opt = torch.optim.AdamW(model.parameters(), lr=LR)

    model.train()
    for ep in range(EPOCHS):
        tot = 0.0
        for ids, am, yb in dl:
            opt.zero_grad()
            logits = model(input_ids=ids.to(DEVICE), attention_mask=am.to(DEVICE)).logits
            loss = lossfn(logits, yb.to(DEVICE))
            loss.backward(); opt.step(); tot += float(loss)
        log.info("    epoch %d/%d loss=%.3f", ep + 1, EPOCHS, tot / max(1, len(dl)))

    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(X_test_txt), EVAL_BATCH):
            ids = enc_te["input_ids"][i:i + EVAL_BATCH].to(DEVICE)
            am = enc_te["attention_mask"][i:i + EVAL_BATCH].to(DEVICE)
            preds.append(model(input_ids=ids, attention_mask=am).logits.argmax(-1).cpu().numpy())
    pred = np.concatenate(preds)
    per = f1_score(y_test, pred, labels=[0, 1, 2, 3], average=None, zero_division=0)
    m = {
        "accuracy": round(float(accuracy_score(y_test, pred)), 4),
        "macro_f1": round(float(f1_score(y_test, pred, average="macro", zero_division=0)), 4),
        "mcc": round(float(matthews_corrcoef(y_test, pred)), 4),
        "per_tier_f1": {TIER_NAMES[i]: round(float(per[i]), 4) for i in range(4)},
        "tier3_recall": round(float(recall_score(y_test, pred, labels=[3], average="macro", zero_division=0)), 4),
    }
    del model; gc.collect()
    return m

def _try_tok(model_id, tkw):
    try:
        AutoTokenizer.from_pretrained(model_id, **tkw); return True
    except Exception:
        return False

log.info("fine-tune routine ready")

## 4. Run the grid (merge into existing results)

In [ ]:
BENCH_PATH = OUTPUT_DIR / "backbone_benchmark_finetuned.json"
existing = {}
if BENCH_PATH.exists():
    try:
        existing = json.loads(BENCH_PATH.read_text(encoding="utf-8")).get("results", {})
    except Exception:
        existing = {}

results = dict(existing)
load_errors = {}
to_run = [(n, mid, g) for (n, mid, g) in ALL_BACKBONES if (not RUN_ONLY or n in RUN_ONLY)]

for name, model_id, gated in to_run:
    if gated and not os.environ.get("HF_TOKEN"):
        load_errors[name] = "gated repo needs HF_TOKEN in .env (not set) — add it and re-run with RUN_ONLY=" + name
        log.warning("SKIP %s — %s", name, load_errors[name]); continue
    t0 = time.time()
    results[name] = {"model_id": model_id, "by_token_cap": {}}
    try:
        for cap in TOKEN_CAPS:
            log.info("fine-tuning %s @%d ...", name, cap)
            te0 = time.time()
            m = finetune_eval(model_id, cap, gated)
            m["train_seconds"] = round(time.time() - te0, 1)
            results[name]["by_token_cap"][str(cap)] = m
            log.info("  %s @%d -> acc=%.3f macroF1=%.3f mcc=%.3f T3rec=%.3f (%.0fs)",
                     name, cap, m["accuracy"], m["macro_f1"], m["mcc"], m["tier3_recall"], m["train_seconds"])
    except Exception as e:
        load_errors[name] = f"{type(e).__name__}: {e}"
        results.pop(name, None)
        log.warning("FAILED %s — %s", name, e)
    log.info("%s done in %.0fs", name, time.time() - t0)

print("ran:", [n for n in to_run if results.get(n[0], {}).get('by_token_cap')], "| errors:", list(load_errors))

## 5. Tables ( backbone comparison +  token sensitivity) + persist (merge)

In [ ]:
PRIMARY = "128"
print("=== FINE-TUNED BACKBONE BENCHMARK (@128 tokens, frozen TEST) ===")
print(f"{'backbone':<12}{'acc':>8}{'macroF1':>9}{'mcc':>8}{'T3rec':>8}")
ranked = []
for name in results:
    m = results[name]["by_token_cap"].get(PRIMARY)
    if m:
        print(f"{name:<12}{m['accuracy']:>8.3f}{m['macro_f1']:>9.3f}{m['mcc']:>8.3f}{m['tier3_recall']:>8.3f}")
        ranked.append((name, m["macro_f1"]))
best = max(ranked, key=lambda x: x[1])[0] if ranked else None
print("best backbone by macro-F1 @128:", best)

token_sens = {}
for name in results:
    caps = results[name]["by_token_cap"]
    if "128" in caps and "256" in caps:
        token_sens[name] = {"f1_128": caps["128"]["macro_f1"], "f1_256": caps["256"]["macro_f1"],
                            "delta_256_minus_128": round(caps["256"]["macro_f1"] - caps["128"]["macro_f1"], 4)}

benchmark = {
    "part": "stage2_part2.3b_backbone_finetuned",
    "version": PART_VERSION,
    "principle": "P1; P3 (train=youtube+synthetic, test=reddit+twitter FROZEN);  backbone;  token cap",
    "method": "actual fine-tuning (AutoModelForSequenceClassification, class-weighted CE), single-seed smoke; "
              "full >=5-seed grid = scale-up",
    "device": DEVICE, "epochs": EPOCHS, "lr": LR, "batch": BATCH, "token_caps": TOKEN_CAPS, "seed": SEED,
    "train_n": len(train_rows), "test_n": len(test_rows),
    "train_tier_dist": {int(t): int((y_train == t).sum()) for t in range(4)},
    "test_tier_dist": {int(t): int((y_test == t).sum()) for t in range(4)},
    "results": results, "load_errors": load_errors, "best_backbone_macro_f1_at128": best,
    "honest_note": "Single-seed, pilot-scale (202 train) fine-tune; weak/variant numbers expected and Tier-3 "
                   "test support is tiny (13). Real publishable benchmark = full >=5-seed grid on the scaled "
                   "corpus. Reported as-measured (P-series honesty).",
}
BENCH_PATH.write_text(json.dumps(benchmark, indent=2, ensure_ascii=False), encoding="utf-8")
(OUTPUT_DIR / "token_sensitivity_finetuned.json").write_text(json.dumps(
    {"metric": "macro_f1", "by_backbone": token_sens,
     "note": "128 vs 256 differs only for posts >128 tokens."}, indent=2, ensure_ascii=False), encoding="utf-8")
log.info("wrote backbone_benchmark_finetuned.json (%d backbones) + token_sensitivity_finetuned.json", len(results))

## 6. Card & This step is complete when

In [ ]:
print("=" * 64); print("Objective 2 · notebook 07b — FINE-TUNED BENCHMARK CARD"); print("=" * 64)
print("backbones with results:", [n for n in results if results[n]['by_token_cap']])
print("load errors           :", list(load_errors) or "none")
print("best @128 (macro-F1)  :", best)
for name in results:
    m = results[name]["by_token_cap"].get("128")
    if m:
        print(f"  {name:<12} acc={m['accuracy']:.3f} macroF1={m['macro_f1']:.3f} mcc={m['mcc']:.3f} "
              f"T3rec={m['tier3_recall']:.3f} perTier={m['per_tier_f1']}")
dod = BENCH_PATH.exists() and len([n for n in results if results[n]['by_token_cap']]) >= 1
print("\nCompletion check:", "PASS" if dod else "INCOMPLETE",
      "(note: IndicBERT pending HF_TOKEN)" if "indicbert" not in results else "")